In [ ]:
#@title ## ▶️ Schritt 1: Initialisierung (einmal ausführen) { display-mode: "form" }
#@markdown Lädt alle Abhängigkeiten, Modelle und Münzbilder. Dauert beim ersten Mal etwas länger.

%%capture --no-display

# ── Imports ──────────────────────────────────
import warnings; warnings.filterwarnings('ignore')
import os, math, ast, numpy as np, cv2, requests
import tensorflow as tf; tf.get_logger().setLevel('ERROR')
from tensorflow import keras
import matplotlib.pyplot as plt, matplotlib.cm as cm, matplotlib.image as mpim
import skimage.transform
from PIL import Image as PILImage
from io import BytesIO
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# ── Modelle herunterladen (nur beim 1. Mal) ──
if not os.path.exists("vgg16_types_95.keras"):
    !wget -q -O vgg16_types.zip 'https://drive.usercontent.google.com/download?id=1m0N1pqY1mF50_XykAZsvk8lJkyXfKlhw&export=download&confirm=t'
    !unzip -qo vgg16_types.zip && rm -f vgg16_types.zip
if not os.path.exists("vgg16_mints_96.keras"):
    !wget -q -O vgg16_mints.zip 'https://drive.usercontent.google.com/download?id=1LRhWPSIgWGcmrCOo2nZzpYfo6q7NC91e&export=download&confirm=t'
    !unzip -qo vgg16_mints.zip && rm -f vgg16_mints.zip
if not os.path.exists("dict_types_95.txt"):
    !wget -q -O dict_types_95.txt 'https://docs.google.com/uc?export=download&id=1lJFaMwvTde_oxxF2FK5jKP4BaYf1Tgzi&confirm=t'
if not os.path.exists("dict_mints_96.txt"):
    !wget -q -O dict_mints_96.txt 'https://docs.google.com/uc?export=download&id=1DwaiC5Ec_-rxhaJVzAyyhEX9yujfW3JM&confirm=t'

# ── Modelle laden ────────────────────────────
model_types = tf.keras.models.load_model('vgg16_types_95.keras')
model_mints = tf.keras.models.load_model('vgg16_mints_96.keras')
with open('dict_types_95.txt') as f: dict_types = ast.literal_eval(f.read())
with open('dict_mints_96.txt') as f: dict_mints = ast.literal_eval(f.read())

IMAGE_SIZE = (224, 224)
LAST_CONV = "block5_conv3"
TEMP = "temp_demo"; os.makedirs(TEMP, exist_ok=True)

# ── Münzdaten ────────────────────────────────
COINS = [
    {"name": "CN Typ 513 – Maroneia",
     "obv_url": "https://data.corpus-nummorum.eu/storage/coins/3454/img/3979/o/thumbnails/lg.jpeg",
     "rev_url": "https://data.corpus-nummorum.eu/storage/coins/3454/img/3979/r/thumbnails/lg.jpeg",
     "obv_desc": "Protome eines springenden Pferdes nach links.",
     "rev_desc": "Sternblume im quadratum incusum.",
     "link": "https://www.corpus-nummorum.eu/types/513"},
    {"name": "CN Typ 8862 – Pergamon",
     "obv_url": "https://data.corpus-nummorum.eu/storage/coins/32574/img/35626/o/thumbnails/lg.jpeg",
     "rev_url": "https://data.corpus-nummorum.eu/storage/coins/32574/img/35626/r/thumbnails/lg.jpeg",
     "obv_desc": "Kopf des Philetairos mit Efeukranz nach rechts.",
     "rev_desc": "Athena nach links thronend, Kranz in der rechten Hand.",
     "link": "https://www.corpus-nummorum.eu/types/8862"},
    {"name": "CN Münze 1775 (Typ 513) – Maroneia",
     "obv_url": "https://data.corpus-nummorum.eu/storage/coins/1775/img/14001/o/thumbnails/lg.jpeg",
     "rev_url": "https://data.corpus-nummorum.eu/storage/coins/1775/img/14001/r/thumbnails/lg.jpeg",
     "obv_desc": "Protome eines springenden Pferdes nach links. Perlkreis.",
     "rev_desc": "Sternblume mit vier Haupt- und drei kleinen Strahlen.",
     "link": "https://www.corpus-nummorum.eu/coins/1775"},
]

# ── Bilder vorladen ──────────────────────────
def _download(url):
    return PILImage.open(BytesIO(requests.get(url, timeout=15).content)).convert("RGB")

for i, c in enumerate(COINS):
    c["obv_img"] = _download(c["obv_url"])
    c["rev_img"] = _download(c["rev_url"])
    obv_p = os.path.join(TEMP, f"c{i}_obv.jpg"); c["obv_img"].save(obv_p); c["obv_path"] = obv_p
    rev_p = os.path.join(TEMP, f"c{i}_rev.jpg"); c["rev_img"].save(rev_p); c["rev_path"] = rev_p

# ── Hilfsfunktionen (kompakt) ────────────────
def _hconcat(imgs):
    h = min(i.shape[0] for i in imgs)
    return cv2.hconcat([cv2.resize(i, (int(i.shape[1]*h/i.shape[0]), h)) for i in imgs])

def _pad_square(img):
    h, w, *_ = img.shape
    mx = max(h, w)
    s = 1
    mn = min(h, w)
    if mn < mx: s = max(1, mn / mn)
    if round(mx * s) > mx: s = mx / mx
    ph = (mx - h) / 2; pw = (mx - w) / 2
    return np.pad(img, ((math.floor(ph), math.ceil(ph)), (math.floor(pw), math.ceil(pw)), (0,0)), mode='constant')

def _combine(obv_p, rev_p):
    o, r = cv2.imread(obv_p), cv2.imread(rev_p)
    c = _hconcat([o, r]) if o.shape[0] != r.shape[0] else cv2.hconcat([o, r])
    h, w, *_ = c.shape; mx, mn = max(h,w), min(h,w)
    dtype = c.dtype; scale = max(1, mn/mn)
    if round(mx*scale) > mx: scale = mx/mx
    if scale != 1: c = skimage.transform.resize(c, (round(h*scale), round(w*scale)), order=1, mode='constant', preserve_range=True).astype(dtype)
    c = _pad_square(c)
    out = os.path.join(TEMP, "combined.jpg"); cv2.imwrite(out, c); return out

def _gradcam(img_array, model):
    gm = tf.keras.models.Model([model.inputs], [model.get_layer(LAST_CONV).output, model.output])
    with tf.GradientTape() as tape:
        conv, preds = gm(img_array); idx = tf.argmax(preds[0]); cc = preds[:, idx]
    g = tape.gradient(cc, conv); p = tf.reduce_mean(g, axis=(0,1,2))
    hm = tf.squeeze(conv[0] @ p[..., tf.newaxis]); hm = tf.maximum(hm, 0) / tf.math.reduce_max(hm)
    return hm.numpy()

def _overlay(img_path, heatmap):
    img = keras.preprocessing.image.img_to_array(keras.preprocessing.image.load_img(img_path))
    jet = cm.get_cmap("jet")(np.arange(256))[:, :3]
    jh = keras.preprocessing.image.array_to_img(jet[np.uint8(255*heatmap)])
    jh = keras.preprocessing.image.img_to_array(jh.resize((img.shape[1], img.shape[0])))
    return keras.preprocessing.image.array_to_img(jh * 0.3 + img)

# ── Übersicht anzeigen ───────────────────────
fig, axes = plt.subplots(1, len(COINS), figsize=(4*len(COINS), 4))
for i, c in enumerate(COINS):
    combo = np.hstack([np.array(c["obv_img"].resize((150,150))), np.array(c["rev_img"].resize((150,150)))])
    axes[i].imshow(combo); axes[i].set_title(f"[{i+1}] {c['name']}", fontsize=10); axes[i].axis('off')
plt.suptitle("Vorausgewählte Münzen", fontsize=14, fontweight='bold'); plt.tight_layout(); plt.show()

# ── Auswahl-Widgets ──────────────────────────
coin_dropdown = widgets.Dropdown(
    options=[(c["name"], i) for i, c in enumerate(COINS)],
    description='Münze:', style={'description_width': '60px'},
    layout=widgets.Layout(width='400px'))
mode_dropdown = widgets.Dropdown(
    options=[("🏛️ Typ-Erkennung", "types"), ("📍 Prägestätten-Erkennung", "mints")],
    description='Modus:', style={'description_width': '60px'},
    layout=widgets.Layout(width='400px'))

display(widgets.HTML("<h3>👇 Münze und Modus wählen, dann Zelle 2 ausführen:</h3>"))
display(widgets.VBox([coin_dropdown, mode_dropdown]))

print("\n✅ Alles bereit! Münze & Modus wählen, dann ▶️ Zelle 2 ausführen.")

In [ ]:
#@title ## ▶️ Schritt 2: Analyse starten { display-mode: "form" }
#@markdown Verwendet die oben gewählte Münze und den Modus.

coin = COINS[coin_dropdown.value]
mode = mode_dropdown.value
model = model_types if mode == "types" else model_mints
mode_label = "Typ" if mode == "types" else "Prägestätte"

print(f"🪙 Analysiere: {coin['name']}")
print(f"📋 Modus: {mode_label}-Erkennung\n")

# Bilder kombinieren & Vorhersage
combined_path = _combine(coin["obv_path"], coin["rev_path"])
preprocess = keras.applications.resnet.preprocess_input
img_array = preprocess(
    np.expand_dims(keras.preprocessing.image.img_to_array(
        keras.preprocessing.image.load_img(combined_path, target_size=IMAGE_SIZE)), axis=0))

preds = model.predict(img_array, verbose=0)
indices = np.argsort(preds[0])[::-1][:5]
top = [(i, preds[0][i]) for i in indices if preds[0][i] > 0]
d = dict_types if mode == "types" else dict_mints
labels = [d[t[0]] for t in top]
probs = [t[1] for t in top]

# GradCAM
model.layers[-1].activation = None
heatmap = _gradcam(img_array, model)
gradcam_img = _overlay(combined_path, heatmap)

# ── Ergebnis-Visualisierung ──────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(coin["obv_img"]); axes[0].set_title("Avers", fontsize=12); axes[0].axis('off')
axes[1].imshow(coin["rev_img"]); axes[1].set_title("Revers", fontsize=12); axes[1].axis('off')
axes[2].imshow(mpim.imread(combined_path)); axes[2].set_title("Kombiniert", fontsize=12); axes[2].axis('off')
axes[3].imshow(gradcam_img); axes[3].set_title("GradCAM", fontsize=12); axes[3].axis('off')
plt.suptitle(f"{mode_label}-Erkennung: {coin['name']}", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# ── Ergebnis-Tabelle ─────────────────────────
print(f"\n{'─'*50}")
print(f"  Top-5 {mode_label}-Vorhersagen")
print(f"{'─'*50}")
medals = ["🥇", "🥈", "🥉", "4.", "5."]
for rank, (label, prob) in enumerate(zip(labels, probs)):
    pct = prob * 100 if prob < 1 else prob
    print(f"  {medals[rank]}  {label:30s}  {pct:6.2f}%")
    if mode == "types":
        print(f"      → https://www.corpus-nummorum.eu/types/{label}")
    else:
        print(f"      → https://www.corpus-nummorum.eu/search/types?type=quicksearch&q={label.replace(' ','+')}")
print(f"{'─'*50}")
print(f"\n🔗 Münze auf CN: {coin['link']}")
print("\n✅ Fertig! Wähle oben eine andere Münze/Modus und führe diese Zelle erneut aus.")